# 03 - Preprocessing dan Split Data
**Proyek:** Hotel Booking Cancellation Prediction | **Tahap:** M4 | **PRD:** v1.1

Tujuan: menyiapkan data yang siap dilatih **tanpa data leakage**.
Urutan penting: bersihkan data → **split dulu** → pelajari transformasi (encoding, scaling, kategori) **hanya dari data training**.

**Fitur (5 input aplikasi):** `lead_time`, `country`, `market_segment`, `total_of_special_requests`, `customer_type`. **Target:** `is_canceled`.
Validasi ulang pemilihan fitur dilakukan di M5-M6 memakai 3 model dan cross-validation.

In [ ]:
import os, glob, json
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

RANDOM_STATE = 42
os.makedirs('data/processed', exist_ok=True); os.makedirs('reports', exist_ok=True)

cands = (glob.glob('/kaggle/input/**/hotel_bookings.csv', recursive=True) + glob.glob('./hotel_bookings.csv')
         + glob.glob('./data/raw/hotel_bookings.csv') + glob.glob('/mnt/user-data/uploads/hotel_bookings.csv'))
assert cands, 'hotel_bookings.csv tidak ditemukan'
raw = pd.read_csv(cands[0])
print('Data asli:', raw.shape)

## 1. Pembersihan data

In [ ]:
# 1) Hapus duplikat (PRD 7.5): semua kolom kecuali reservation_status_date
cols_dup = [c for c in raw.columns if c != 'reservation_status_date']
df = raw.drop_duplicates(subset=cols_dup).reset_index(drop=True)
print('Setelah hapus duplikat:', df.shape, '| dihapus:', len(raw) - len(df))

# 2) Ambil hanya fitur terpilih + target. reservation_status*, parking, dll. tidak dipakai (leakage / bukan input).
CAT_FEATS = ['country', 'market_segment', 'customer_type']
NUM_FEATS = ['lead_time', 'total_of_special_requests']
FEATURES = NUM_FEATS + CAT_FEATS
TARGET = 'is_canceled'
data = df[FEATURES + [TARGET]].copy()

print('\nMissing value pada fitur terpilih:')
print(data.isna().sum())
print('\nNilai tidak wajar (harus tidak ada):')
print('lead_time < 0:', (data.lead_time < 0).sum(), '| special_requests < 0:', (data.total_of_special_requests < 0).sum())
print('market_segment langka:'); print(data.market_segment.value_counts().tail(3))

**Keputusan:**
- **Missing value:** hanya `country` yang kosong (451 baris, ≈0,5%). Tidak dibuang, diberi kategori `Unknown` di dalam pipeline (jadi saat aplikasi menerima negara kosong tidak error).
- **Nilai tidak wajar:** nilai ekstrem pada `adults`, `adr`, atau tamu = 0 tidak perlu dibersihkan karena kolom itu **tidak dipakai** sebagai fitur. Baris tetap dipertahankan agar tidak membuang data tanpa alasan kuat.
- **Outlier `lead_time`** (maksimum 737 hari): tidak dibuang karena pemesanan jauh-jauh hari adalah data yang valid. Model pohon tidak sensitif terhadap outlier; untuk Logistic Regression dipakai transformasi `log1p` untuk meredam kemiringan.

## 2. Split train/test (dilakukan sebelum transformasi apa pun)

In [ ]:
X = data[FEATURES]; y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

split_info = pd.DataFrame({
    'jumlah baris': [len(X_train), len(X_test)],
    '% batal': [round(y_train.mean()*100, 2), round(y_test.mean()*100, 2)]},
    index=['train', 'test'])
display(split_info)
assert len(set(X_train.index) & set(X_test.index)) == 0, 'Ada baris yang sama di train dan test!'
print('Tidak ada baris yang sama di train dan test: OK')

pd.concat([X_train, y_train], axis=1).to_csv('data/processed/train.csv', index=True)
pd.concat([X_test, y_test], axis=1).to_csv('data/processed/test.csv', index=True)

**Interpretasi:** split 80:20 *stratified* menjaga proporsi kelas batal tetap sama pada train dan test (≈27%), sehingga evaluasi tidak bias. `random_state=42` membuat hasil dapat direproduksi. Data test **disimpan terpisah dan tidak disentuh** sampai evaluasi akhir (M6).

## 3. Pipeline preprocessing

In [ ]:
# lead_time sangat miring ke kanan (maks 737 hari): untuk Logistic Regression dipakai log1p lalu standardisasi
# (log1p tidak punya parameter yang dipelajari dari data, jadi aman dari leakage)
log_tf = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', max_categories=None, sparse_output=False)),
])
# country: pertahankan 10 negara teratas di train, sisanya (termasuk Unknown) digabung ke 'infrequent'
country_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', max_categories=11, sparse_output=False)),
])

# Pipeline untuk model pohon (Random Forest, XGBoost): tanpa scaling
prep_tree = ColumnTransformer([
    ('country', country_pipe, ['country']),
    ('cat', cat_pipe, ['market_segment', 'customer_type']),
    ('num', 'passthrough', NUM_FEATS),
])

# Pipeline untuk Logistic Regression: log1p(lead_time) + standardisasi
prep_linear = ColumnTransformer([
    ('country', country_pipe, ['country']),
    ('cat', cat_pipe, ['market_segment', 'customer_type']),
    ('lead', Pipeline([('log', log_tf), ('scale', StandardScaler())]), ['lead_time']),
    ('req', StandardScaler(), ['total_of_special_requests']),
])

# fit HANYA pada data training
prep_tree.fit(X_train); prep_linear.fit(X_train)
Xt_tree = prep_tree.transform(X_train); Xt_lin = prep_linear.transform(X_train)
print('Bentuk X_train hasil transformasi -> tree:', Xt_tree.shape, '| linear:', Xt_lin.shape)

names = prep_tree.get_feature_names_out()
print(len(names), 'fitur setelah encoding:'); print(list(names))

**Alasan tiap langkah (siapkan untuk tanya jawab):**
- **Imputer `Unknown`:** nilai kosong pada `country` dijadikan kategori sendiri, bukan dihapus.
- **One-Hot Encoding:** `country`, `market_segment`, `customer_type` adalah kategori tanpa urutan, sehingga tidak boleh diberi angka 1,2,3 (model akan menganggap ada urutan).
- **`max_categories=11` pada `country`:** hanya 10 negara teratas di data training yang diberi kolom sendiri, sisanya (termasuk `Unknown`) digabung ke satu kolom "lainnya". Ini mencegah ±170 kolom jarang dan cocok dengan dropdown aplikasi (top-10 + Lainnya). Jumlah 10 dapat diubah saat validasi ulang di M6.
- **`log1p` + scaling hanya untuk Logistic Regression:** regresi linier sensitif pada skala fitur, sedangkan Random Forest dan XGBoost berbasis pohon sehingga tidak butuh scaling.
- **Fit hanya di train:** rata-rata, simpangan baku, dan daftar kategori dipelajari dari data training saja, lalu diterapkan ke test. Inilah yang mencegah leakage.

## 4. Pengecekan hasil transformasi

In [ ]:
# Kategori yang dipelajari dari TRAIN
oh = prep_tree.named_transformers_['country'].named_steps['onehot']
kept = [c for c in oh.categories_[0] if c not in (oh.infrequent_categories_[0] if oh.infrequent_categories_[0] is not None else [])]
print('Negara dengan kolom sendiri (%d):' % len(kept), kept)
print('Negara digabung ke "lainnya":', len(oh.infrequent_categories_[0]))
print('market_segment:', list(prep_tree.named_transformers_['cat'].named_steps['onehot'].categories_[0]))
print('customer_type :', list(prep_tree.named_transformers_['cat'].named_steps['onehot'].categories_[1]))

# Statistik hasil scaling pada TRAIN (harus mean~0, std~1)
lin_out = pd.DataFrame(Xt_lin, columns=prep_linear.get_feature_names_out())
print('\nMean & std kolom ter-scaling (train):')
print(lin_out[['lead__lead_time','req__total_of_special_requests']].agg(['mean','std']).round(3))

# Uji ketahanan: input dengan negara baru/kosong tidak boleh error (ini yang terjadi di aplikasi)
sample = pd.DataFrame([{'lead_time': 120, 'total_of_special_requests': 1, 'country': 'XXX',
                        'market_segment': 'Online TA', 'customer_type': 'Transient'},
                       {'lead_time': 5, 'total_of_special_requests': 0, 'country': np.nan,
                        'market_segment': 'Direct', 'customer_type': 'Contract'}])
print('\nUji input baru -> bentuk:', prep_tree.transform(sample).shape, '(tidak error: OK)')

# Cek tidak ada NaN setelah transformasi
assert not np.isnan(prep_tree.transform(X_test)).any() and not np.isnan(prep_linear.transform(X_test)).any()
print('Transformasi test tanpa NaN: OK')

**Interpretasi:** setelah fit di train, kolom ter-scaling punya rata-rata ≈ 0 dan simpangan baku ≈ 1. Input dengan negara yang belum pernah terlihat (`XXX`) atau kosong tetap bisa ditransformasi tanpa error, karena digabung ke kategori "lainnya". Ini penting karena pengguna aplikasi bisa memilih negara apa pun. Transformasi pada data test tidak menghasilkan NaN.

## 5. Menyimpan konfigurasi

In [ ]:
config = {'features': FEATURES, 'num_feats': NUM_FEATS, 'cat_feats': CAT_FEATS, 'target': TARGET,
          'random_state': RANDOM_STATE, 'test_size': 0.2,
          'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
          'market_segment_values': sorted(X_train.market_segment.unique().tolist()),
          'customer_type_values': sorted(X_train.customer_type.unique().tolist()),
          'top_countries': X_train.country.value_counts().head(10).index.tolist()}
with open('reports/preprocessing_config.json', 'w') as f: json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 6. Kesimpulan M4
1. Duplikat dihapus (32.239 baris) → 87.151 baris; hanya 5 fitur + target yang dipakai.
2. Split 80:20 stratified (`random_state=42`) dilakukan **sebelum** transformasi; test disimpan terpisah.
3. Dua pipeline: `prep_tree` (Random Forest, XGBoost) dan `prep_linear` (Logistic Regression, dengan log1p dan scaling), semuanya di-fit hanya pada train.
4. `country` kosong → `Unknown`; hanya top-10 negara diberi kolom sendiri; input baru tidak membuat error.
5. Class imbalance (≈27% batal) belum ditangani dengan resampling; mulai dari `class_weight` di model dan dievaluasi di M5-M6.

**Langkah berikutnya: M5 Modeling** (`04_model_training.ipynb`): baseline Logistic Regression, Random Forest, dan XGBoost (perlu `pip install xgboost scikit-learn joblib`), dilatih pada data training dengan Pipeline, lalu dievaluasi.